In [3]:
!pip3 install pandas numpy

  Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.6-cp314-cp314-macosx_14_0_arm64.whl (5.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [ ]:
import pandas as pd 
import numpy as np

In [77]:
def sort_to_loss_train():
    import os
    from pathlib import Path
    
    unzipped_path: Path = os.path.join(os.getcwd() + '/unzipped')
    
    if not os.path.exists(unzipped_path):
        print(f"Error: The directory '{unzipped_path}' does not exist.")
        print("You must do: create ZIP -> unzip the new samples")
        return
    srt = {"train": [], "loss": [], 'neural_structure': []} 
    
    for i,name in enumerate(os.listdir(unzipped_path)):
        match name[:1]:  
            case 'l': 
                #print(f"{i} loss")
                srt["loss"].append(os.path.join(unzipped_path, name))
            case 't':
                #print(f"{i} train")  
                srt["train"].append(os.path.join(unzipped_path, name))
            case 'N':
                srt['neural_structure'].append(os.path.join(unzipped_path, name))
            case _ as e:
                print("I had to stope the loop - something unexpected happended")
                print(f"error: {e}") 
                exit(2)
    return srt
sort_to_loss_train()

I had to stope the loop - something unexpected happended
error: .


{'train': ['/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_11.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_10.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_9.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_8.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_6.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_7.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_5.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_4.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_1.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_3.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_2.csv'],
 'loss': ['/Users/samuel/Documents/git/neuralparser/unzipped/loss_history_net_9.csv',
  '/Users/s

In [78]:
import pandas as pd
import numpy as np 
import pathlib
from pathlib import Path

csvs = sort_to_loss_train()
loss_csvs = csvs['loss']
train_csvs = csvs['train']

def to_path(df):
    if not isinstance(df,list):
        raise ValueError("Can't convert string to deref")
    
    tp = pathlib.Path(*df[:1])
    return tp

dft = pd.read_csv(to_path(train_csvs))
dfT = pd.read_csv(to_path(loss_csvs))

pd.set_option('display.max_colwidth', None)
print(f"Neural Len: {dft.shape}")
print("Info: ", dft.head(n=3))

I had to stope the loop - something unexpected happended
error: .
Neural Len: (10000, 2)
Info:     predicted_digit  confidence_score
0                0          0.494351
1                1          0.641322
2                2          0.247405


In [118]:
def to_path(df):
    if not isinstance(df,list):
        raise ValueError(f"Can't convert {type(df)} to deref")
    
    tp = pathlib.Path(*df[:1])
    return tp

import re
import pandas as pd

def parse_to_math(sliced):
    sequential_token = re.compile(r'Sequential\(')
    linear_token = re.compile(r'\(?\d+\)?:Linear\(in_features=(\d+),out_features=(\d+),bias=(True|False)\)')
    relu_token = re.compile(r'\(?\d+\)?:ReLU\(\)')
    
    parsed_symbols = []
    
    # 2. Iterate and match against tokens
    for line in sliced:
        if sequential_token.search(line):
            # Track the start of the sequence container block
            parsed_symbols.append("SEQ[")
            
        elif linear_token.match(line):
            match = linear_token.match(line)
            in_f = match.group(1)   # Grabs 784, 512, etc.
            out_f = match.group(2)  # Grabs 512, 10, etc.
            bias = match.group(3)   # Grabs True/False
            
            # Format as requested: L(in, out, bias)
            parsed_symbols.append(f"L({in_f},{out_f},b={bias})")
            
        elif relu_token.match(line):
            parsed_symbols.append("R()")
            
        elif line == ')':
            parsed_symbols.append("]")

    # 3. Join the processed symbols into a readable stream path
    return parsed_symbols

def read_and_parse(): 
    f = sort_to_loss_train()['neural_structure']
    o = pd.read_csv(to_path(f))
    for i in range(0, o['architecture'].size):
        sliced = o['architecture'][i].replace(' ', '').split('\n')
        print(parse_to_math(sliced))
        
    #print(parse_to_math(o[0]))

read_and_parse()

I had to stope the loop - something unexpected happended
error: .
['SEQ[', 'L(784,512,b=True)', 'R()', 'L(512,10,b=True)', ']']
['SEQ[', 'L(784,256,b=True)', 'R()', 'L(256,10,b=True)', ']']
['SEQ[', 'L(784,128,b=True)', 'R()', 'L(128,10,b=True)', ']']
['SEQ[', 'L(784,512,b=True)', 'R()', 'L(512,256,b=True)', 'R()', 'L(256,10,b=True)', ']']
['SEQ[', 'L(784,256,b=True)', 'R()', 'L(256,128,b=True)', 'R()', 'L(128,10,b=True)', ']']
['SEQ[', 'L(784,512,b=True)', 'R()', 'L(512,128,b=True)', 'R()', 'L(128,10,b=True)', ']']
['SEQ[', 'L(784,512,b=True)', 'R()', 'L(512,256,b=True)', 'R()', 'L(256,128,b=True)', 'R()', 'L(128,10,b=True)', ']']
['SEQ[', 'L(784,256,b=True)', 'R()', 'L(256,256,b=True)', 'R()', 'L(256,256,b=True)', 'R()', 'L(256,10,b=True)', ']']
['SEQ[', 'L(784,128,b=True)', 'R()', 'L(128,64,b=True)', 'R()', 'L(64,10,b=True)', ']']
['SEQ[', 'L(784,256,b=True)', 'R()', 'L(256,64,b=True)', 'R()', 'L(64,10,b=True)', ']']
['SEQ[', 'L(784,512,b=True)', 'R()', 'L(512,64,b=True)', 'R()', 'L

In [ ]:
#$$\text{Estimated Accuracy} = (784 \times 512 \times W_1) + (512 \times B_1) + R \times ((512 \times 10 \times W_2) + (10 \times B_2))$$